In [1]:
import pandas as pd
import numpy as np

In [2]:
df=pd.read_csv("../data/raw/telecom_customer_churn.csv")

In [3]:
df_preprocessed=df.copy()

In [4]:
df_preprocessed.isnull().sum()

CustomerID                0
Gender                 2454
SeniorCitizen             0
Partner                2819
Dependents             3298
Tenure                 2069
PhoneService              0
MultipleLines          3787
InternetService        1971
OnlineSecurity         6515
OnlineBackup           5297
DeviceProtection       4548
TechSupport            7211
StreamingTV            6045
StreamingMovies        6875
Contract               2243
PaperlessBilling       3036
PaymentMethod          3555
MonthlyCharges         2635
TotalCharges           3032
MonthlyDataUsageGB     8619
NumberOfComplaints    10091
SatisfactionScore      7448
AvgCallDuration        9123
Churn                     0
dtype: int64

In [5]:
df_preprocessed.duplicated().sum()

np.int64(900)

In [6]:
df_preprocessed = df_preprocessed.drop_duplicates()

In [7]:
df_preprocessed.duplicated().sum()

np.int64(0)

In [8]:
df_preprocessed = df_preprocessed.drop(columns=["CustomerID"])

In [9]:
x = df_preprocessed.drop(columns=["Churn"])
y = df_preprocessed["Churn"]

In [10]:
from sklearn.model_selection import train_test_split

In [11]:
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42)

In [12]:
print(x_train.shape)
print(x_test.shape)
print(y_train.shape)
print(y_test.shape)

(47280, 23)
(11820, 23)
(47280,)
(11820,)


In [13]:
num_data = [
    "SeniorCitizen",
    "Tenure",
    "MonthlyCharges",
    "TotalCharges",
    "MonthlyDataUsageGB",
    "NumberOfComplaints",
    "SatisfactionScore",
    "AvgCallDuration"
]

In [14]:
cat_data = [
    "Gender",
    "Partner",
    "Dependents",
    "PhoneService",
    "MultipleLines",
    "InternetService",
    "OnlineSecurity",
    "OnlineBackup",
    "DeviceProtection",
    "TechSupport",
    "StreamingTV",
    "StreamingMovies",
    "Contract",
    "PaperlessBilling",
    "PaymentMethod"
]

In [15]:
from sklearn.impute import SimpleImputer

imp_cat_data = SimpleImputer(strategy="most_frequent")
x_train[cat_data] = imp_cat_data.fit_transform(x_train[cat_data])
x_test[cat_data] = imp_cat_data.transform(x_test[cat_data])

In [16]:
imp_num_data = SimpleImputer(strategy="mean")
x_train[num_data] = imp_num_data.fit_transform(x_train[num_data])
x_test[num_data] = imp_num_data.transform(x_test[num_data])

In [17]:
print(x_train.isnull().sum().sum())
print(x_test.isnull().sum().sum())

0
0


In [18]:
import pickle

with open("../artifacts/cat_imputer.pkl", "wb") as file:
    pickle.dump(imp_cat_data, file)

with open("../artifacts/num_imputer.pkl", "wb") as file:
    pickle.dump(imp_num_data, file)

In [19]:
from sklearn.preprocessing import OneHotEncoder

In [20]:
encoder = OneHotEncoder(drop="first", handle_unknown="ignore", sparse_output=False)

In [21]:
x_train_cat = encoder.fit_transform(x_train[cat_data])

In [22]:
x_test_cat = encoder.transform(x_test[cat_data])

In [39]:
import pickle

with open("../artifacts/encoder.pkl", "wb") as file:
    pickle.dump(encoder, file)

In [24]:
from sklearn.preprocessing import StandardScaler

In [25]:
scaler = StandardScaler()

In [26]:
x_train_num = scaler.fit_transform(x_train[num_data])

In [27]:
x_test_num = scaler.transform(x_test[num_data])

In [40]:
with open("../artifacts/scaler.pkl", "wb") as file:
    pickle.dump(scaler, file)

In [29]:
x_train_final = np.hstack([x_train_num, x_train_cat])

In [30]:
x_test_final = np.hstack([x_test_num, x_test_cat])

In [31]:
cat_columns = encoder.get_feature_names_out(cat_data)

In [32]:
final_columns = list(num_data) + list(cat_columns)

In [33]:
x_train_final = pd.DataFrame(
    x_train_final,
    columns=final_columns,
    index=x_train.index
)

In [34]:
x_train_final.head()

,SeniorCitizen,Tenure,MonthlyCharges,TotalCharges,MonthlyDataUsageGB,NumberOfComplaints,SatisfactionScore,AvgCallDuration,Gender_Male,Partner_Yes,...,StreamingTV_No internet service,StreamingTV_Yes,StreamingMovies_No internet service,StreamingMovies_Yes,Contract_One year,Contract_Two year,PaperlessBilling_Yes,PaymentMethod_Credit card,PaymentMethod_Electronic check,PaymentMethod_Mailed check
50656,-0.966713,-1.553737,-1.297456,-1.176345,0.086808,-0.346637,-1.676111,0.217980,0.0,1.0,...,1.0,0.0,1.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0
12506,1.034433,0.622167,0.170643,0.641761,1.385442,2.060974,1.311434,0.147376,1.0,0.0,...,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
19159,-0.966713,-0.151487,-0.534991,-0.425790,-1.223027,-0.002693,0.937991,-0.616428,1.0,0.0,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0
1134,1.034433,-0.586668,0.000000,-0.852120,-1.447058,-0.346637,-0.182338,1.604382,1.0,0.0,...,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0
13852,-0.966713,0.380400,0.118047,0.276437,0.552793,1.373085,-0.929225,-1.129910,1.0,0.0,...,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0


In [35]:
x_test_final = pd.DataFrame(
    x_test_final,
    columns=final_columns,
    index=x_test.index
)

In [36]:
x_test_final.head()

,SeniorCitizen,Tenure,MonthlyCharges,TotalCharges,MonthlyDataUsageGB,NumberOfComplaints,SatisfactionScore,AvgCallDuration,Gender_Male,Partner_Yes,...,StreamingTV_No internet service,StreamingTV_Yes,StreamingMovies_No internet service,StreamingMovies_Yes,Contract_One year,Contract_Two year,PaperlessBilling_Yes,PaymentMethod_Credit card,PaymentMethod_Electronic check,PaymentMethod_Mailed check
17548,-0.966713,0.000000,0.000000,-1.303012,0.000000,0.000000,-3.316841e-16,2.280318e-16,0.0,1.0,...,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0
48454,-0.966713,0.912288,-0.923283,0.133080,-1.754727,0.341252,-1.302668e+00,-1.617718e+00,1.0,1.0,...,1.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,1.0
3960,1.034433,1.154055,-0.608060,0.342104,0.000000,-1.722414,-3.316841e-16,2.280318e-16,1.0,1.0,...,0.0,0.0,0.0,1.0,0.0,1.0,1.0,0.0,0.0,0.0
26403,1.034433,-0.441608,1.590734,0.350869,0.341457,0.341252,1.911048e-01,2.280318e-16,1.0,1.0,...,1.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0
15330,1.034433,0.235340,1.617914,1.444455,0.000000,0.341252,-5.557814e-01,-1.065725e+00,0.0,1.0,...,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0


both training + testing data in one CSV

In [37]:
train_data = pd.concat([x_train_final, y_train], axis=1)
test_data = pd.concat([x_test_final, y_test], axis=1)

final_data = pd.concat([train_data, test_data])

In [38]:
final_data.to_csv(
    "../data/processed/telecom_customer_processed.csv",
    index=False
)